# FD001 Exploratory Data Analysis

**Prerequisites:** Run `python -m src.data --download` before executing these cells.

This notebook covers:
1. Dataset dimensions and engine counts
2. Operating settings analysis
3. Sensor distribution plots
4. Sensor correlation heatmap
5. Engine degradation trajectories
6. RUL distribution (raw and capped)
7. Constant sensor identification
8. Missing-value analysis
9. Train vs. test comparison


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ensure project root is on path.
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import ROOT, load_train, load_test

FIGDIR = ROOT / 'reports' / 'figures'
FIGDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
RUL_CAP = 125

print('Loading FD001 …')
train = load_train('FD001')
test, test_labels = load_test('FD001')
SENSORS = [f'sensor_{i}' for i in range(1, 22)]
SETTINGS = ['setting_1', 'setting_2', 'setting_3']
print('Done.')

## 1. Dataset dimensions and engine counts

In [ ]:
summary = {
    'Training rows': len(train),
    'Training engines': train['engine_id'].nunique(),
    'Avg cycles per training engine': round(train.groupby('engine_id')['cycle'].max().mean(), 1),
    'Min training trajectory length': int(train.groupby('engine_id')['cycle'].max().min()),
    'Max training trajectory length': int(train.groupby('engine_id')['cycle'].max().max()),
    'Test rows': len(test),
    'Test engines': test['engine_id'].nunique(),
    'Official test labels': len(test_labels),
    'Missing values (train)': int(train.isna().sum().sum()),
    'Missing values (test)': int(test.isna().sum().sum()),
}
for k, v in summary.items():
    print(f'  {k}: {v}')

## 2. Operating settings analysis

In [ ]:
print('Operating settings — unique values per column:')
for s in SETTINGS:
    uniq = train[s].nunique()
    print(f'  {s}: {uniq} unique values  (range [{train[s].min():.4g}, {train[s].max():.4g}])')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, SETTINGS):
    ax.hist(train[col], bins=40, edgecolor='white')
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
fig.suptitle('FD001 — Operating Setting Distributions', fontweight='bold')
fig.tight_layout()
fig.savefig(FIGDIR / 'operating_settings.png', dpi=160)
plt.show()
print('FD001 has one operating condition → settings cluster tightly.')

## 3. Sensor distribution plots

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(20, 16), constrained_layout=True)
flat_axes = axes.ravel()
for i, sensor in enumerate(SENSORS):
    ax = flat_axes[i]
    ax.hist(train[sensor].dropna(), bins=40, edgecolor='none', alpha=0.8)
    ax.set_title(sensor, fontsize=10)
    ax.set_xlabel('Value', fontsize=8)
    ax.set_ylabel('Count', fontsize=8)
# Hide unused subplot.
for j in range(len(SENSORS), len(flat_axes)):
    flat_axes[j].set_visible(False)
fig.suptitle('FD001 — Sensor Value Distributions (all training rows)', fontsize=14, fontweight='bold')
fig.savefig(FIGDIR / 'sensor_distributions.png', dpi=160)
plt.show()

## 4. Constant sensor identification

In [ ]:
constant_sensors = [s for s in SENSORS if train[s].nunique() <= 1]
low_variance = [s for s in SENSORS if 1 < train[s].nunique() <= 5]
print(f'Constant sensors (single unique value): {constant_sensors}')
print(f'Very low variance sensors (≤5 unique values): {low_variance}')
print()
print('Sensor value ranges (may reveal informative vs. constant):')
print(train[SENSORS].describe().T[['mean', 'std', 'min', 'max']].to_string())

## 5. Sensor correlation heatmap

In [ ]:
corr = train[SENSORS].corr()
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Pearson correlation')
ax.set_xticks(range(len(SENSORS)))
ax.set_yticks(range(len(SENSORS)))
ax.set_xticklabels(SENSORS, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(SENSORS, fontsize=9)
ax.set_title('Sensor Correlation Heatmap — FD001 Training Set', fontweight='bold', pad=12)
fig.tight_layout()
fig.savefig(FIGDIR / 'sensor_correlation.png', dpi=160)
plt.show()

# Most correlated pairs.
corr_abs = corr.abs()
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape, dtype=bool), k=1))
top_pairs = upper.stack().sort_values(ascending=False).head(5)
print('Top 5 most correlated sensor pairs:')
print(top_pairs.to_string())

## 6. Engine degradation trajectories

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9), constrained_layout=True)
informative_sensors = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_11', 'sensor_12']
engines_to_plot = train['engine_id'].unique()[:8]

for ax, sensor in zip(axes.ravel(), informative_sensors):
    for eng_id in engines_to_plot:
        d = train[train['engine_id'] == eng_id]
        ax.plot(d['cycle'], d[sensor], alpha=0.7, linewidth=0.9)
    ax.set_xlabel('Cycle')
    ax.set_ylabel('Sensor value')
    ax.set_title(f'{sensor} degradation trajectories')

fig.suptitle('FD001 — Engine Degradation Trajectories (8 engines)', fontsize=13, fontweight='bold')
fig.savefig(FIGDIR / 'degradation_trajectories.png', dpi=160)
plt.show()

## 7. RUL distribution (raw vs. capped)

In [ ]:
y_raw = train['RUL']
y_capped = y_raw.clip(upper=RUL_CAP)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(y_raw, bins=50, edgecolor='none', alpha=0.85, color='steelblue')
axes[0].set_title('Training RUL — Raw')
axes[0].set_xlabel('RUL (cycles)')
axes[0].set_ylabel('Row count')
axes[0].axvline(RUL_CAP, color='red', linestyle='--', label=f'Cap = {RUL_CAP}')
axes[0].legend()

axes[1].hist(y_capped, bins=50, edgecolor='none', alpha=0.85, color='darkorange')
axes[1].set_title(f'Training RUL — Capped at {RUL_CAP}')
axes[1].set_xlabel('RUL (cycles)')
axes[1].set_ylabel('Row count')

axes[2].hist(test_labels, bins=30, edgecolor='none', alpha=0.85, color='green')
axes[2].set_title('Official Test RUL Labels')
axes[2].set_xlabel('RUL (cycles)')
axes[2].set_ylabel('Engine count')

fig.suptitle('FD001 — RUL Distributions', fontweight='bold')
fig.tight_layout()
fig.savefig(FIGDIR / 'rul_distribution.png', dpi=160)
plt.show()

print(f'Training RUL — min: {y_raw.min()}, max: {y_raw.max()}, mean: {y_raw.mean():.1f}')
print(f'Official test RUL — min: {test_labels.min():.0f}, max: {test_labels.max():.0f}, mean: {test_labels.mean():.1f}')

## 8. Training vs. test trajectory length comparison

In [ ]:
train_lengths = train.groupby('engine_id')['cycle'].max()
test_lengths = test.groupby('engine_id')['cycle'].max()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(train_lengths, bins=25, edgecolor='white', alpha=0.85, color='steelblue')
axes[0].set_title('Training: Trajectory lengths per engine')
axes[0].set_xlabel('Last observed cycle (run to failure)')
axes[0].set_ylabel('Engine count')
axes[1].hist(test_lengths, bins=25, edgecolor='white', alpha=0.85, color='darkorange')
axes[1].set_title('Test: Trajectory lengths per engine (truncated)')
axes[1].set_xlabel('Last observed cycle (not at failure)')
axes[1].set_ylabel('Engine count')
fig.suptitle('FD001 — Training vs. Test Trajectory Lengths', fontweight='bold')
fig.tight_layout()
fig.savefig(FIGDIR / 'trajectory_lengths.png', dpi=160)
plt.show()

print(f'Training: {train_lengths.mean():.1f} cycles mean, {train_lengths.min()}–{train_lengths.max()} range')
print(f'Test:     {test_lengths.mean():.1f} cycles mean, {test_lengths.min()}–{test_lengths.max()} range')

## 9. Missing value analysis

In [ ]:
print('Missing values per column in training set:')
mv = train.isna().sum()
print(mv[mv > 0].to_string() if mv.sum() > 0 else '  None — dataset is complete.')

print('\nMissing values per column in test set:')
mv_test = test.isna().sum()
print(mv_test[mv_test > 0].to_string() if mv_test.sum() > 0 else '  None — dataset is complete.')

print('\nNote: Early-cycle rolling standard deviations will be NaN (expected).')
print('These are handled by median imputation inside each pipeline fold.')